In [1]:
import pandas as pd
import os
import numpy as np
import re
from datetime import datetime, timedelta

In [2]:
pd.set_option('display.max_columns', None)

# **Step 1. Merge Normal patients with EHR**
### output: <span style="color:DodgerBlue;">**normal_ehr** & **normal_ehr_@index_1**</span>

#### <span style="color:#FF6347;">**READ**</span> normal_candidate file

In [3]:
# file_path = "P:/Dataset/MO-DBT-data-curation/Cancer/mo_cancer.xlsx"
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "normal_candidate" + ".xlsx")
df = pd.read_excel(file_path)

In [4]:
df.head(3)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,EXAM_NAME,Study,Series,SIDE,Index,time_to_INDEX_days,time_to_INDEX_months
0,4330000534,67674008,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-06-03,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,L,INDEX-1,387.0,12.7
1,4330000534,67674008,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-06-03,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,R,INDEX-1,387.0,12.7
2,4330000534,451457076,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-06-25,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,L,INDEX,NaN,NaN


In [5]:
df.columns

Index(['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'COMPOSITION_NAME',
       'FINDING_CATEGORY', 'FINDING_REC', 'EXAM_COMPLETED_DATE',
       'duplicate_count', 'EXAM_NAME', 'Study', 'Series', 'SIDE', 'Index',
       'time_to_INDEX_days', 'time_to_INDEX_months'],
      dtype='object')

In [6]:
extract_cols = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE', 'Study', 'Index']

In [7]:
df_patient_visit = df.drop_duplicates(
    subset=extract_cols, keep = 'first').reset_index(drop = True)[extract_cols]

In [8]:
df_patient_visit.head(5)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,EXAM_COMPLETED_DATE,Study,Index
0,4330000534,67674008,2021-06-03,SCREEN,INDEX-1
1,4330000534,451457076,2022-06-25,SCREEN,INDEX
2,4330002818,66443471,2021-03-25,SCREEN,INDEX-1
3,4330002818,451338997,2022-06-23,SCREEN,INDEX
4,4330002818,451184127,2022-06-29,NaN,INDEX


#### <span style="color:#FF6347;">**READ**</span> **EHR** (hormonal_mens, risk_factors, patient_demo, vitals )

In [9]:
# hormonal_mens = pd.read_excel(os.path.join("P:/Dataset/MO-DBT-data-curation/", "Cancer", "Cleaned", "hormonal_mens" + ".xlsx"))
# risk_factors = pd.read_excel(os.path.join("P:/Dataset/MO-DBT-data-curation/", "Cancer", "Cleaned", "risk_factors" + ".xlsx"))
# patient_demo = pd.read_excel(os.path.join("P:/Dataset/MO-DBT-data-curation/", "Cancer", "Cleaned", "patient_demo" + ".xlsx"))
# vitals = pd.read_excel(os.path.join("P:/Dataset/MO-DBT-data-curation/", "Cancer", "Cleaned", "vitals" + ".xlsx"))

hormonal_mens = pd.read_excel(os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "Cleaned", "hormonal_mens" + ".xlsx"))
risk_factors = pd.read_excel(os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "Cleaned", "risk_factors" + ".xlsx"))
patient_demo = pd.read_excel(os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "Cleaned", "patient_demo" + ".xlsx"))
vitals = pd.read_excel(os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "Cleaned", "vitals" + ".xlsx"))

### **1. Merge BMI from vitals**

In [10]:
vitals.head(5)

,PATIENT_STUDY_ID,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI
0,4330000534,2017-08-30,2480.00,OZ,65.0,IN,25.79
1,4330000534,2018-01-03,2480.00,OZ,65.0,IN,25.79
2,4330000534,2018-03-14,2476.80,OZ,65.0,IN,25.76
3,4330000534,2018-04-04,2222.24,OZ,65.0,IN,23.11
4,4330000534,2018-04-13,2433.88,OZ,65.0,IN,25.31


In [11]:
df_patient_visit['EXAM_COMPLETED_DATE'] = pd.to_datetime(df_patient_visit['EXAM_COMPLETED_DATE'])
vitals['DATE_TAKEN']                    = pd.to_datetime(vitals['DATE_TAKEN'])

visit_sorted  = df_patient_visit.sort_values('EXAM_COMPLETED_DATE').reset_index(drop=True)
vitals_sorted = vitals.sort_values('DATE_TAKEN').reset_index(drop=True)

df_patient_visit_vital = pd.merge_asof(
    visit_sorted,
    vitals_sorted,
    left_on  = 'EXAM_COMPLETED_DATE',
    right_on = 'DATE_TAKEN',
    by       = 'PATIENT_STUDY_ID',
    # tolerance=pd.Timedelta(days=365),
    direction= 'nearest'
)

print(f"df_patient_visit : {len(df_patient_visit)} rows")
print(f"df_patient_visit_vital : {len(df_patient_visit_vital)} rows")

df_patient_visit : 60763 rows
df_patient_visit_vital : 60763 rows


In [12]:
df_patient_visit_vital.sort_values(['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'], ignore_index=True, inplace=True)

In [13]:
df_patient_visit_vital.head(5)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,EXAM_COMPLETED_DATE,Study,Index,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI
0,4330000534,67674008,2021-06-03,SCREEN,INDEX-1,2021-06-29,2976.0,OZ,65.0,IN,30.95
1,4330000534,451457076,2022-06-25,SCREEN,INDEX,2022-04-19,2848.0,OZ,65.0,IN,29.62
2,4330002818,66443471,2021-03-25,SCREEN,INDEX-1,2021-08-03,2912.0,OZ,NaN,IN,28.29
3,4330002818,451184127,2022-06-29,NaN,INDEX,2022-07-28,3156.0,OZ,67.5,IN,30.44
4,4330002818,451338997,2022-06-23,SCREEN,INDEX,2022-05-26,3044.0,OZ,67.0,IN,29.80


### **2. Merge hormonal_mens, risk_factors, patient_demo**

In [14]:
merge_hr = hormonal_mens.merge(risk_factors, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE'], how='inner')
merge_hrp = merge_hr.merge(patient_demo, on=['PATIENT_STUDY_ID'], how='inner')

### **2.1. Calculate Age at exam date**

#### <span style="color:darkcyan">**Function**</span>

In [15]:
from pandas import Int64Dtype
def calculate_age_at_study(df):
    """
    Calculates the patient's age in years at the time of the study 
    based on the 'PatientBirthDate' and 'StudyDate' columns.
    """
    
    # 1. Ensure date columns are in datetime format
    # The format 'YYYY-MM-DD' is used for parsing
    try:
        df['BIRTH_DATE_DT'] = pd.to_datetime(df['BIRTH_DATE'], format='%Y-%m-%d')
        # Note: The StudyDate column often includes time (e.g., '2020-06-01 09:10:52').
        # We can let pandas infer the format for this one since it's cleaner.
        df['EXAM_COMPLETED_DATE_DT'] = pd.to_datetime(df['EXAM_COMPLETED_DATE'], errors='coerce') 
    except ValueError as e:
        print(f"Error parsing date format: {e}. Please check your date column formats.")
        return df

    # 2. Calculate the difference in days
    time_difference = df['EXAM_COMPLETED_DATE_DT'] - df['BIRTH_DATE_DT']
    
    # 3. Convert the difference into whole years (integer format)
    # The .dt.days attribute gives the number of days, which is divided by 365.25 
    # and then explicitly cast to an integer to capture only the full years elapsed.
    mask = df['BIRTH_DATE_DT'].notna() & df['EXAM_COMPLETED_DATE_DT'].notna()
    df.loc[mask, 'PatientAge'] = (time_difference[mask].dt.days / 365.25).astype(int)
    # df['PatientAge'] = (time_difference.dt.days / 365.25).astype('Int64')
    
    # Clean up the intermediate columns
    
    df = df.drop(columns=['BIRTH_DATE_DT', 'EXAM_COMPLETED_DATE_DT'])
    
    return df

#### <span style="color:blue">**Main**</span>

In [16]:
merge_hrp = calculate_age_at_study(merge_hrp)

In [17]:
merge_hrp.head(3)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,MENSTRUAL_STATUS_CD,EXAM_COMPLETED_DATE,RISK_FACTOR_No family history of breast cancer,RISK_FACTOR_Personal breast cancer history,"RISK_FACTOR_Weak family history of breast cancer (aunt, grandmother, cousin)",RISK_FACTOR_Late child bearing (after 30),"RISK_FACTOR_Very strong family history of breast cancer (mother, sister, daughter, pre-menopause or multiple post-menopausal first degree relatives)","RISK_FACTOR_Family history of ovarian cancer in mother, sister, or daughter",RISK_FACTOR_Post-menopausal patient,RISK_FACTOR_BRCA1 gene mutation,RISK_FACTOR_Nulliparous,RISK_FACTOR_Family history of ovarian cancer in distant blood relatives,"RISK_FACTOR_Intermediate family history of breast cancer (mother, sister, daughter, post-menopause)","RISK_FACTOR_History of high risk lesion on previous biopsy, LCIS, atypical hyperplasia",RISK_FACTOR_Previous chest radiation therapy,RISK_FACTOR_Other family history of breast cancer,RISK_FACTOR_BRCA2 gene mutation,RISK_FACTOR_Family history unknown for breast cancer,RISK_FACTOR_History of ovarian cancer,RISK_FACTOR_History of endometrial cancer,RISK_FACTOR_Not available,BIRTH_DATE,GENDER_TITLE,RACE_KOREAN,RACE_WHITE,RACE_INDIAN (ASIAN),RACE_BLACK,RACE_UNREPORTED/CHOSE NOT TO DISCLOSE,RACE_OTHER ASIAN,RACE_VIETNAMESE,RACE_CHINESE,RACE_AMERICAN INDIAN/ALASKA NATIVE,RACE_FILIPINO,RACE_OTHER,RACE_JAPANESE,RACE_HAWAIIAN,RACE_OTHER PACIFIC ISLANDER,ETHNIC_NOT HISPANIC OR LATINO,"ETHNIC_ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN",ETHNIC_UNREPORTED/CHOSE NOT TO DISCLOSE,ETHNIC_HISPANIC OR LATINO,"ETHNIC_MEXICAN, MEXICAN AMERICAN, CHICANO",PatientAge
0,4330000534,78048053,13,32,0,0,0,0,3.0,3.0,NaN,PRE,2018-03-28,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1973-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,44.0
1,4330000534,64787403,13,32,0,0,0,0,3.0,3.0,NaN,PRE,2019-05-18,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1973-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,45.0
2,4330000534,63433254,13,32,0,0,0,0,3.0,3.0,NaN,PRE,2019-06-20,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1973-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,45.0


### **3. Merge MO cancer patients with EHR**

In [18]:
merge_hrp['EXAM_COMPLETED_DATE'] = pd.to_datetime(merge_hrp['EXAM_COMPLETED_DATE'])

In [19]:
df_patient_visit_ehr = pd.merge(df_patient_visit_vital, merge_hrp, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE'], how='left')

In [20]:
df_patient_visit_ehr

,PATIENT_STUDY_ID,ACCESSION_NUMBER,EXAM_COMPLETED_DATE,Study,Index,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,MENSTRUAL_STATUS_CD,RISK_FACTOR_No family history of breast cancer,RISK_FACTOR_Personal breast cancer history,"RISK_FACTOR_Weak family history of breast cancer (aunt, grandmother, cousin)",RISK_FACTOR_Late child bearing (after 30),"RISK_FACTOR_Very strong family history of breast cancer (mother, sister, daughter, pre-menopause or multiple post-menopausal first degree relatives)","RISK_FACTOR_Family history of ovarian cancer in mother, sister, or daughter",RISK_FACTOR_Post-menopausal patient,RISK_FACTOR_BRCA1 gene mutation,RISK_FACTOR_Nulliparous,RISK_FACTOR_Family history of ovarian cancer in distant blood relatives,"RISK_FACTOR_Intermediate family history of breast cancer (mother, sister, daughter, post-menopause)","RISK_FACTOR_History of high risk lesion on previous biopsy, LCIS, atypical hyperplasia",RISK_FACTOR_Previous chest radiation therapy,RISK_FACTOR_Other family history of breast cancer,RISK_FACTOR_BRCA2 gene mutation,RISK_FACTOR_Family history unknown for breast cancer,RISK_FACTOR_History of ovarian cancer,RISK_FACTOR_History of endometrial cancer,RISK_FACTOR_Not available,BIRTH_DATE,GENDER_TITLE,RACE_KOREAN,RACE_WHITE,RACE_INDIAN (ASIAN),RACE_BLACK,RACE_UNREPORTED/CHOSE NOT TO DISCLOSE,RACE_OTHER ASIAN,RACE_VIETNAMESE,RACE_CHINESE,RACE_AMERICAN INDIAN/ALASKA NATIVE,RACE_FILIPINO,RACE_OTHER,RACE_JAPANESE,RACE_HAWAIIAN,RACE_OTHER PACIFIC ISLANDER,ETHNIC_NOT HISPANIC OR LATINO,"ETHNIC_ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN",ETHNIC_UNREPORTED/CHOSE NOT TO DISCLOSE,ETHNIC_HISPANIC OR LATINO,"ETHNIC_MEXICAN, MEXICAN AMERICAN, CHICANO",PatientAge
0,4330000534,67674008,2021-06-03,SCREEN,INDEX-1,2021-06-29,2976.0,OZ,65.0,IN,30.95,13.0,32.0,0.0,0.0,0.0,0.0,3.0,3.0,NaN,PRE,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1973-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,47.0
1,4330000534,451457076,2022-06-25,SCREEN,INDEX,2022-04-19,2848.0,OZ,65.0,IN,29.62,13.0,32.0,0.0,0.0,0.0,0.0,3.0,3.0,NaN,PRE,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1973-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,48.0
2,4330002818,66443471,2021-03-25,SCREEN,INDEX-1,2021-08-03,2912.0,OZ,NaN,IN,28.29,13.0,25.0,0.0,0.0,0.0,0.0,3.0,5.0,2019-09-25,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1977-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,43.0
3,4330002818,451184127,2022-06-29,NaN,INDEX,2022-07-28,3156.0,OZ,67.5,IN,30.44,13.0,25.0,0.0,0.0,0.0,0.0,3.0,5.0,2019-09-25,PRE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1977-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,44.0
4,4330002818,451338997,2022-06-23,SCREEN,INDEX,2022-05-26,3044.0,OZ,67.0,IN,29.80,13.0,25.0,0.0,0.0,0.0,0.0,3.0,5.0,2019-09-25,PRE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1977-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,44.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60837,4339978786,76855223,2019-01-07,SCREEN,INDEX-1,NaT,NaN,NaN,NaN,NaN,NaN,13.0,20.0,41.0,0.0,0.0,0.0,2.0,2.0,NaN,POSTNAT,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1953-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,65.0
60838,4339982047,450915296,2022-10-12,SCREEN,INDEX,2022-07-27,79.0,KG,Na

#### <span style="color:#FF6347;">**SAVE**</span> **normal_ehr, normal_ehr_@_index_1**

In [21]:
df_patient_visit_ehr["DATE_TAKEN"] = df_patient_visit_ehr["DATE_TAKEN"].dt.strftime("%Y-%m-%d")
df_patient_visit_ehr["EXAM_COMPLETED_DATE"] = df_patient_visit_ehr["EXAM_COMPLETED_DATE"].dt.strftime("%Y-%m-%d")

In [22]:
df_patient_visit_ehr['PATIENT_STUDY_ID'].nunique()

30047

In [23]:
df_patient_ehr = df_patient_visit_ehr[df_patient_visit_ehr["Index"]=="INDEX-1"]

In [24]:
df_patient_ehr['PATIENT_STUDY_ID'].nunique()

30047

In [25]:
# output_file = os.path.join("P:/Dataset/MO-DBT-data-curation/", "Cancer", 'mo_cancer_ehr' + ".xlsx")
# df_patient_visit_ehr.to_excel(output_file, index=False)

# output_file = os.path.join("P:/Dataset/MO-DBT-data-curation/", "Cancer", 'mo_cancer_ehr_@_index_1' + ".xlsx")
# df_patient_ehr.to_excel(output_file, index=False)


output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", 'normal_ehr' + ".xlsx")
df_patient_visit_ehr.to_excel(output_file, index=False)

output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", 'normal_ehr_@_index_1' + ".xlsx")
df_patient_ehr.to_excel(output_file, index=False)